In [7]:
import pandas as pd
import numpy as np

feat_path = "/home/nzheludkov/phd/ml-critical-path-prediction/dataset/usb_phy/features/endpoint_features_place.csv"
lab_path  = "/home/nzheludkov/phd/ml-critical-path-prediction/dataset/usb_phy/features/endpoint_slacks_postroute.csv"

feat = pd.read_csv(feat_path)
lab  = pd.read_csv(lab_path)
df = feat.merge(lab[["endpoint", "worst_slack_ns"]], on="endpoint", how="inner").copy()

df["crit_cont"] = -df["worst_slack_ns"]  # higher => more critical

drop_cols = ["endpoint", "instance", "worst_pin_regex", "D_net", "worst_slack_ns"]
base = df.drop(columns=[c for c in drop_cols if c in df.columns]).copy()

cat_cols = [c for c in ["ff_ref", "ff_orient", "ff_status"] if c in base.columns]
X = pd.get_dummies(base.drop(columns=["crit_cont"]), columns=cat_cols, dummy_na=True)

rng = np.random.default_rng(42)
idx = np.arange(len(df))
rng.shuffle(idx)
n_train = int(0.8 * len(df))
train_idx = idx[:n_train]
val_idx   = idx[n_train:]

X_train = X.iloc[train_idx].to_numpy()
X_val   = X.iloc[val_idx].to_numpy()

crit_train = df["crit_cont"].to_numpy()[train_idx]
crit_val   = df["crit_cont"].to_numpy()[val_idx]

def to_relevance_grades(crit):
    # integer grades in [0..N-1], most critical => largest grade
    order = np.argsort(crit)  # least->most
    grades = np.empty_like(order, dtype=np.int32)
    grades[order] = np.arange(len(crit), dtype=np.int32)
    return grades

y_train = to_relevance_grades(crit_train)
y_val   = to_relevance_grades(crit_val)

group_train = [len(train_idx)]
group_val   = [len(val_idx)]

from xgboost import XGBRanker

ranker = XGBRanker(
    objective="rank:pairwise",
    n_estimators=400,
    learning_rate=0.08,
    max_depth=5,
    min_child_weight=1.0,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    random_state=42,
    tree_method="hist",
    eval_metric="ndcg",
    ndcg_exp_gain=False,
)

ranker.fit(
    X_train, y_train,
    group=group_train,
    eval_set=[(X_val, y_val)],
    eval_group=[group_val],
    verbose=False,
)

val_pred = ranker.predict(X_val)
val_slack = df["worst_slack_ns"].to_numpy()[val_idx]
val_endpoints = df["endpoint"].to_numpy()[val_idx]

def recall_at_k(y_true_slack, y_score, k):
    if k <= 0:
        return np.nan
    order = np.argsort(-y_score)[:k]
    violating = (y_true_slack < 0).astype(int)
    denom = int(violating.sum())
    if denom == 0:
        return np.nan
    return float(violating[order].sum() / denom)

def ndcg_at_k(rel_true, y_score, k):
    if k <= 0:
        return np.nan
    order = np.argsort(-y_score)[:k]
    rel = rel_true[order].astype(float)
    gains = (2.0 ** rel - 1.0)
    discounts = 1.0 / np.log2(np.arange(2, 2 + len(rel)))
    dcg = float(np.sum(gains * discounts))

    ideal = np.argsort(-rel_true)[:k]
    rel_i = rel_true[ideal].astype(float)
    gains_i = (2.0 ** rel_i - 1.0)
    idcg = float(np.sum(gains_i * discounts))
    return float(dcg / idcg) if idcg > 0 else np.nan

def spearman_rho(a, b):
    def rankdata(x):
        order = np.argsort(x)
        ranks = np.empty_like(order, dtype=float)
        ranks[order] = np.arange(1, len(x) + 1)
        _, inv, counts = np.unique(x, return_inverse=True, return_counts=True)
        for i, c in enumerate(counts):
            if c > 1:
                pos = np.where(inv == i)[0]
                ranks[pos] = ranks[pos].mean()
        return ranks
    ra = rankdata(a); rb = rankdata(b)
    ra -= ra.mean(); rb -= rb.mean()
    denom = np.sqrt((ra**2).sum() * (rb**2).sum())
    return float((ra * rb).sum() / denom) if denom > 0 else np.nan

K = int((val_slack < 0).sum())  # #violating endpoints in val

# heuristics
val_D_hpwl = df["D_hpwl"].to_numpy()[val_idx]
val_fanout = df["D_fanout_proxy"].to_numpy()[val_idx]

def eval_model(name, score):
    return {
        "model": name,
        "Recall@K": recall_at_k(val_slack, score, K),
        "NDCG@K": ndcg_at_k(crit_val, score, max(K, 1)),
        "Spearman": spearman_rho(crit_val, score),
    }

res_df = pd.DataFrame([
    eval_model("XGBRanker(pairwise)", val_pred),
    eval_model("Heuristic: D_hpwl", val_D_hpwl),
    eval_model("Heuristic: D_fanout_proxy", val_fanout),
])

topn = min(12, len(val_idx))
top_order = np.argsort(-val_pred)[:topn]
top_table = pd.DataFrame({
    "endpoint": val_endpoints[top_order],
    "pred_score": val_pred[top_order],
    "true_slack_ns": val_slack[top_order],
    "D_hpwl": val_D_hpwl[top_order],
    "D_fanout_proxy": val_fanout[top_order],
}).sort_values("pred_score", ascending=False)

print("Merged dataset size:", df.shape)
print(f"Train: {len(train_idx)} endpoints | Val: {len(val_idx)} endpoints")
print(f"Violating endpoints in Val (slack<0): K={K}")
print("\nMetrics (on Val):")
display(res_df)

print("\nTop predicted critical endpoints (Val):")
display(top_table)


Merged dataset size: (105, 43)
Train: 84 endpoints | Val: 21 endpoints
Violating endpoints in Val (slack<0): K=1

Metrics (on Val):


,model,Recall@K,NDCG@K,Spearman
0,XGBRanker(pairwise),0.0,-0.835842,0.441558
1,Heuristic: D_hpwl,0.0,-3.611515,0.110390
2,Heuristic: D_fanout_proxy,0.0,-3.611515,-0.580124



Top predicted critical endpoints (Val):


,endpoint,pred_score,true_slack_ns,D_hpwl,D_fanout_proxy
0,_446_/D,1.084316,0.347165,16.507,1
1,_457_/D,0.516542,1.122111,5.526,1
2,_473_/D,0.449468,2.572467,4.568,1
3,_444_/D,0.328702,0.383096,9.189,1
4,_450_/D,0.252750,1.231972,4.604,1
5,_460_/D,-0.401678,0.163129,4.574,1
6,_491_/D,-0.682777,-0.328695,11.048,1
7,_449_/D,-1.038747,3.434534,30.765,4
8,_515_/D,-1.061034,2.388063,2.986,1
9,_485_/D,-1.108407,1.958855,3.875,1


In [6]:
import pandas as pd
import numpy as np
from xgboost import XGBRanker

FEAT = "/home/nzheludkov/phd/ml-critical-path-prediction/dataset/mc_top/features/endpoint_features_place.csv"
LAB  = "/home/nzheludkov/phd/ml-critical-path-prediction/dataset/mc_top/features/endpoint_slacks_postroute.csv"

# 1) Load + merge
feat = pd.read_csv(FEAT)
lab  = pd.read_csv(LAB)
df = feat.merge(lab[["endpoint", "worst_slack_ns"]], on="endpoint", how="inner").copy()

# 2) Continuous criticality (for evaluation)
df["crit_cont"] = -df["worst_slack_ns"]   # higher => more critical

# 3) Drop identifiers / high-card text
drop_cols = ["endpoint", "instance", "worst_pin_regex", "D_net", "worst_slack_ns"]
base = df.drop(columns=[c for c in drop_cols if c in df.columns]).copy()

# 4) One-hot low-card categoricals
cat_cols = [c for c in ["ff_ref", "ff_orient", "ff_status"] if c in base.columns]
X = pd.get_dummies(base.drop(columns=["crit_cont"]), columns=cat_cols, dummy_na=True)
crit = base["crit_cont"].to_numpy()

# 5) Train/val split (single block for now)
rng = np.random.default_rng(42)
idx = np.arange(len(df))
rng.shuffle(idx)
n_train = int(0.8 * len(df))
tr = idx[:n_train]
va = idx[n_train:]

X_tr = X.iloc[tr].to_numpy()
X_va = X.iloc[va].to_numpy()
crit_tr = crit[tr]
crit_va = crit[va]

# 6) Convert continuous criticality -> integer relevance grades (required by XGB 3.x)
def to_grades(crit_vec):
    order = np.argsort(crit_vec)               # least -> most
    g = np.empty_like(order, dtype=np.int32)
    g[order] = np.arange(len(crit_vec), dtype=np.int32)
    return g

y_tr = to_grades(crit_tr)
y_va = to_grades(crit_va)

group_tr = [len(tr)]   # one query per split
group_va = [len(va)]

# 7) Train ranker
ranker = XGBRanker(
    objective="rank:pairwise",
    n_estimators=300,
    learning_rate=0.08,
    max_depth=5,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    tree_method="hist",
    n_jobs=4,
    random_state=42,
    eval_metric="ndcg",
    ndcg_exp_gain='false',
)

ranker.fit(
    X_tr, y_tr,
    group=group_tr,
    eval_set=[(X_va, y_va)],
    eval_group=[group_va],
    verbose=False
)

# 8) Predict scores (higher => more critical)
pred_va = ranker.predict(X_va)

# -------- Metrics you want --------
slack_va = df["worst_slack_ns"].to_numpy()[va]

def recall_at_k(slack_true, score, k):
    if k <= 0:
        return np.nan
    order = np.argsort(-score)[:k]
    violating = (slack_true < 0).astype(int)
    denom = int(violating.sum())
    return float(violating[order].sum() / denom) if denom > 0 else np.nan

def spearman_rho(a, b):
    # small helper without scipy
    def rankdata(x):
        order = np.argsort(x)
        r = np.empty_like(order, dtype=float)
        r[order] = np.arange(1, len(x)+1)
        _, inv, cnt = np.unique(x, return_inverse=True, return_counts=True)
        for i, c in enumerate(cnt):
            if c > 1:
                pos = np.where(inv == i)[0]
                r[pos] = r[pos].mean()
        return r
    ra, rb = rankdata(a), rankdata(b)
    ra -= ra.mean(); rb -= rb.mean()
    denom = np.sqrt((ra**2).sum() * (rb**2).sum())
    return float((ra*rb).sum()/denom) if denom > 0 else np.nan

K = int((slack_va < 0).sum())   # number of violating endpoints in val
print("Val size:", len(va), "K (#violating):", K)
print("Recall@K:", recall_at_k(slack_va, pred_va, K))
print("Spearman:", spearman_rho(crit_va, pred_va))


Val size: 121 K (#violating): 39
Recall@K: 0.6153846153846154
Spearman: 0.4317627951694824


In [2]:
import pandas as pd
import numpy as np
from xgboost import XGBRanker

# ----------------------------
# CONFIG
# ----------------------------
DATASET_ROOT = "/home/nzheludkov/phd/ml-critical-path-prediction/dataset"

TRAIN_BLOCKS = ["pcm_slv_top", "FIR_filter", "usb_phy"]
TEST_BLOCKS  = ["spi_top"]

import os
import pandas as pd

def load_block(block_name, dataset_root):
    feat_path = os.path.join(
        dataset_root, block_name, "features", "endpoint_features_place.csv"
    )
    lab_path = os.path.join(
        dataset_root, block_name, "features", "endpoint_slacks_postroute.csv"
    )

    feat = pd.read_csv(feat_path)
    lab  = pd.read_csv(lab_path)

    df = feat.merge(
        lab[["endpoint", "worst_slack_ns", "endpoint_clock"]],
        on="endpoint",
        how="inner"
    )

    df["block_name"] = block_name
    return df

# ----------------------------
# LOAD + MERGE
# ----------------------------
feat = pd.read_csv("endpoint_features_place.csv")
lab  = pd.read_csv("endpoint_slacks_postroute.csv")

df = feat.merge(
    lab[["endpoint", "worst_slack_ns", "block_name", "endpoint_clock"]],
    on="endpoint",
    how="inner"
)

# Continuous criticality (for evaluation only)
df["crit_cont"] = -df["worst_slack_ns"]

# ----------------------------
# FEATURE PREP
# ----------------------------
drop_cols = [
    "endpoint", "instance", "D_net", "worst_pin_regex",
    "worst_slack_ns", "crit_cont",
    "block_name", "endpoint_clock"
]

base = df.drop(columns=[c for c in drop_cols if c in df.columns])

cat_cols = [c for c in ["ff_ref", "ff_orient", "ff_status"] if c in base.columns]
X_all = pd.get_dummies(base, columns=cat_cols, dummy_na=True)

# ----------------------------
# SPLIT BY BLOCK
# ----------------------------
train_df = df[df["block_name"].isin(TRAIN_BLOCKS)].copy()
test_df  = df[df["block_name"].isin(TEST_BLOCKS)].copy()

X_train = X_all.loc[train_df.index]
X_test  = X_all.loc[test_df.index]

crit_train = train_df["crit_cont"].to_numpy()
crit_test  = test_df["crit_cont"].to_numpy()

# ----------------------------
# BUILD RANKING GROUPS
# group = (block, clock)
# ----------------------------
def build_groups(df_sub):
    groups = []
    order  = []
    for (_, _), g in df_sub.groupby(["block_name", "endpoint_clock"]):
        groups.append(len(g))
        order.extend(g.index.tolist())
    return groups, order

train_groups, train_order = build_groups(train_df)
test_groups,  test_order  = build_groups(test_df)

X_train = X_train.loc[train_order].to_numpy()
X_test  = X_test.loc[test_order].to_numpy()

crit_train = df.loc[train_order, "crit_cont"].to_numpy()
crit_test  = df.loc[test_order,  "crit_cont"].to_numpy()
slack_test = df.loc[test_order,  "worst_slack_ns"].to_numpy()

# ----------------------------
# CONVERT CONTINUOUS -> GRADES
# (required by XGB >= 3.x)
# ----------------------------
def to_grades_per_group(crit, groups):
    grades = np.empty_like(crit, dtype=np.int32)
    start = 0
    for g in groups:
        end = start + g
        order = np.argsort(crit[start:end])  # least -> most critical
        grades[start:end][order] = np.arange(g)
        start = end
    return grades

y_train = to_grades_per_group(crit_train, train_groups)
y_test  = to_grades_per_group(crit_test,  test_groups)

# ----------------------------
# TRAIN RANKER
# ----------------------------
ranker = XGBRanker(
    objective="rank:pairwise",
    n_estimators=400,
    max_depth=5,
    learning_rate=0.08,
    subsample=0.9,
    colsample_bytree=0.9,
    tree_method="hist",
    random_state=42,
    eval_metric="ndcg",
)

ranker.fit(
    X_train, y_train,
    group=train_groups,
    eval_set=[(X_test, y_test)],
    eval_group=[test_groups],
    verbose=False
)

# ----------------------------
# EVALUATION
# ----------------------------
pred_test = ranker.predict(X_test)

def recall_at_k(slack, score, k):
    if k <= 0:
        return np.nan
    order = np.argsort(-score)[:k]
    violating = (slack < 0).astype(int)
    return violating[order].sum() / violating.sum()

def spearman_rho(a, b):
    return pd.Series(a).corr(pd.Series(b), method="spearman")

K = int((slack_test < 0).sum())

print("TEST BLOCKS:", TEST_BLOCKS)
print("Violating endpoints:", K)
print("Recall@K:", recall_at_k(slack_test, pred_test, K))
print("Spearman:", spearman_rho(crit_test, pred_test))


FileNotFoundError: [Errno 2] No such file or directory: 'endpoint_features_place.csv'

In [7]:
# ============================================================
# XGBoost Learning-to-Rank for Timing-Critical Endpoints
# Multi-block training with explicit train/test blocks,
# Leave-One-Block-Out table + Ablations (incl. spatial)
# ============================================================

import os
import re
import numpy as np
import pandas as pd

from xgboost import XGBRanker

# ----------------------------
# 0) CONFIG
# ----------------------------
DATASET_ROOT = "/home/nzheludkov/phd/ml-critical-path-prediction/dataset"

# Block names are folder names under DATASET_ROOT
BLOCKS = ["pcm_slv_top", "FIR_filter", "spi_top", "usb_phy"]  # add more here

# File names inside each block folder
FEAT_REL = os.path.join("features", "endpoint_features_place.csv")
LAB_REL  = os.path.join("features", "endpoint_slacks_postroute.csv")

# Categorical columns (if present) to one-hot encode
CAT_COLS = ["ff_ref", "ff_orient", "ff_status"]

# Drop columns that should never become ML features (if present)
DROP_ALWAYS = [
    "endpoint", "instance", "D_net", "worst_pin_regex",
    "worst_slack_ns", "crit_cont",
    "block_name", "endpoint_clock"
]

# Spatial features detection regex (edit if your names differ)
SPATIAL_REGEX = r"(?:^|_)ff_neighbors_|neighbors"

# Heuristic columns (if present)
HEUR_HPWL_COL = "D_hpwl"
HEUR_FANOUT_COL = "D_fanout_proxy"

# XGBoost params (reasonable starting point)
XGB_PARAMS = dict(
    objective="rank:pairwise",
    n_estimators=600,
    max_depth=5,
    learning_rate=0.06,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    tree_method="hist",
    eval_metric="ndcg",
    random_state=42,
    n_jobs=8,
    ndcg_exp_gain=False,
)

# Optional: save outputs for paper
SAVE_RESULTS = True
OUT_DIR = "./xgb_rank_results"
os.makedirs(OUT_DIR, exist_ok=True)


# ----------------------------
# 1) LOADERS
# ----------------------------
def load_block(block_name: str, root: str) -> pd.DataFrame:
    feat_path = os.path.join(root, block_name, FEAT_REL)
    lab_path  = os.path.join(root, block_name, LAB_REL)

    feat = pd.read_csv(feat_path)
    lab  = pd.read_csv(lab_path)

    # Expect labels to contain endpoint + worst_slack_ns.
    # endpoint_clock should be present in labels; if not, we’ll fallback later.
    needed = ["endpoint", "worst_slack_ns"]
    for c in needed:
        if c not in lab.columns:
            raise ValueError(f"[{block_name}] Missing column in labels CSV: {c}")

    # If endpoint_clock missing, create a single dummy clock domain
    if "endpoint_clock" not in lab.columns:
        lab = lab.copy()
        lab["endpoint_clock"] = "clk0"

    df = feat.merge(
        lab[["endpoint", "worst_slack_ns", "endpoint_clock"]],
        on="endpoint",
        how="inner"
    )
    df["block_name"] = block_name
    return df


def load_blocks(block_list, root) -> pd.DataFrame:
    dfs = []
    for b in block_list:
        dfs.append(load_block(b, root))
    out = pd.concat(dfs, ignore_index=True)
    return out


# ----------------------------
# 2) FEATURE BUILDING (global)
# Ensure train/test share identical feature space (OHE done jointly)
# ----------------------------
def build_feature_matrix(all_df: pd.DataFrame):
    all_df = all_df.copy()
    all_df["crit_cont"] = -all_df["worst_slack_ns"]  # higher => more critical

    # Keep only feature columns (drop IDs/targets)
    drop_cols = [c for c in DROP_ALWAYS if c in all_df.columns]
    base = all_df.drop(columns=drop_cols, errors="ignore")

    cat_cols = [c for c in CAT_COLS if c in base.columns]
    X_all = pd.get_dummies(base, columns=cat_cols, dummy_na=True)

    return all_df, X_all


# ----------------------------
# 3) GROUPING for ranking
# group = endpoints in one (block_name, endpoint_clock)
# ----------------------------
def build_groups_and_order(df_sub: pd.DataFrame):
    groups = []
    order = []
    for (_, _), g in df_sub.groupby(["block_name", "endpoint_clock"], sort=False):
        groups.append(len(g))
        order.extend(g.index.tolist())
    return groups, order


# ----------------------------
# 4) XGB >= 3.x requires integer, non-negative labels for rank objectives
# Convert continuous criticality -> integer grades per group
# ----------------------------
def to_grades_per_group(crit_vec: np.ndarray, groups: list[int], max_grade: int = 31) -> np.ndarray:
    """
    Convert continuous criticality to integer relevance grades in [0..max_grade]
    for each query group. Preserves ordering approximately (via quantiles).

    max_grade=31 is required when XGBoost uses exponential NDCG gain (default).
    """
    grades = np.empty_like(crit_vec, dtype=np.int32)
    start = 0
    for g in groups:
        end = start + g
        c = crit_vec[start:end]

        # rank positions 0..g-1 (0 least critical, g-1 most critical)
        order = np.argsort(c)
        r = np.empty(g, dtype=np.int32)
        r[order] = np.arange(g, dtype=np.int32)

        # map rank -> [0..max_grade]
        if g == 1:
            grades[start:end] = 0
        else:
            grades[start:end] = np.floor(r * (max_grade / (g - 1))).astype(np.int32)

        start = end
    return grades

# ----------------------------
# 5) METRICS (EDA-friendly)
# ----------------------------
def recall_at_k(slack_true: np.ndarray, score: np.ndarray, k: int) -> float:
    if k <= 0:
        return float("nan")
    violating = (slack_true < 0).astype(int)
    denom = int(violating.sum())
    if denom == 0:
        return float("nan")
    order = np.argsort(-score)[:k]
    return float(violating[order].sum() / denom)


def spearman_rho(a: np.ndarray, b: np.ndarray) -> float:
    return float(pd.Series(a).corr(pd.Series(b), method="spearman"))


def ndcg_at_k(rel_true: np.ndarray, score: np.ndarray, k: int) -> float:
    """
    rel_true: higher => more relevant/critical. We use crit_cont = -slack (continuous).
    """
    if k <= 0:
        return float("nan")
    k = min(k, len(score))
    order = np.argsort(-score)[:k]
    rel = rel_true[order].astype(float)

    gains = (2.0 ** rel - 1.0)
    discounts = 1.0 / np.log2(np.arange(2, 2 + k))
    dcg = float(np.sum(gains * discounts))

    ideal = np.argsort(-rel_true)[:k]
    rel_i = rel_true[ideal].astype(float)
    gains_i = (2.0 ** rel_i - 1.0)
    idcg = float(np.sum(gains_i * discounts))

    return float(dcg / idcg) if idcg > 0 else float("nan")


# ----------------------------
# 6) RUN ONE EXPERIMENT: train_blocks -> test_blocks
# Includes heuristics + optional ablation (feature masks)
# ----------------------------
def run_experiment(all_df: pd.DataFrame, X_all: pd.DataFrame,
                   train_blocks: list[str], test_blocks: list[str],
                   feature_mode: str = "full"):
    """
    feature_mode in {"full", "no_spatial", "only_baseline"}
    - full: all features
    - no_spatial: drop spatial neighbor features
    - only_baseline: keep only a small baseline feature subset (hpwl/fanout proxies etc.)
    """
    # Split by block
    train_df = all_df[all_df["block_name"].isin(train_blocks)].copy()
    test_df  = all_df[all_df["block_name"].isin(test_blocks)].copy()

    if len(train_df) == 0 or len(test_df) == 0:
        raise ValueError(f"Empty split: train={train_blocks}, test={test_blocks}")

    # Feature selection (column mask)
    cols = list(X_all.columns)

    if feature_mode == "full":
        use_cols = cols

    elif feature_mode == "no_spatial":
        use_cols = [c for c in cols if not re.search(SPATIAL_REGEX, c)]
        if len(use_cols) == 0:
            raise ValueError("no_spatial removed all features; check SPATIAL_REGEX")

    elif feature_mode == "only_baseline":
        # Minimal baseline: try to keep hpwl/fanout proxies and a few simple counts if present
        baseline_keys = [
            "D_hpwl", "D_fanout_proxy",
            "D_bbox_dx", "D_bbox_dy", "D_bbox_area",
            "D_term_count", "D_iterms", "D_bterms",
            "ports_cnt",
        ]
        use_cols = [c for c in cols if any(c == k or c.startswith(k + "_") for k in baseline_keys)]
        if len(use_cols) == 0:
            # fallback: keep everything numeric (still a baseline-ish)
            use_cols = cols

    else:
        raise ValueError("feature_mode must be one of: full, no_spatial, only_baseline")

    # Build ranking groups + order
    train_groups, train_order = build_groups_and_order(train_df)
    test_groups,  test_order  = build_groups_and_order(test_df)

    # Align features + targets in group-concatenated order
    X_train = X_all.loc[train_order, use_cols].to_numpy()
    X_test  = X_all.loc[test_order,  use_cols].to_numpy()

    crit_train = all_df.loc[train_order, "crit_cont"].to_numpy()
    crit_test  = all_df.loc[test_order,  "crit_cont"].to_numpy()
    slack_test = all_df.loc[test_order,  "worst_slack_ns"].to_numpy()

    # Grades for XGB training/eval
    y_train = to_grades_per_group(crit_train, train_groups)
    y_test  = to_grades_per_group(crit_test,  test_groups)

    # Train
    ranker = XGBRanker(**XGB_PARAMS)
    ranker.fit(
        X_train, y_train,
        group=train_groups,
        eval_set=[(X_test, y_test)],
        eval_group=[test_groups],
        verbose=False
    )

    # Predict
    pred = ranker.predict(X_test)

    # Metrics (global over test set)
    # In paper, better is: compute metrics per (block,clock) group then average.
    # But start with global: simple and stable.
    K = int((slack_test < 0).sum())
    res = {
        "train_blocks": ",".join(train_blocks),
        "test_blocks": ",".join(test_blocks),
        "feature_mode": feature_mode,
        "N_test_endpoints": int(len(test_df)),
        "N_test_groups": int(len(test_groups)),
        "K_violating": int(K),
        "Recall@K": recall_at_k(slack_test, pred, K),
        "NDCG@K": ndcg_at_k(crit_test, pred, max(K, 1)),
        "Spearman": spearman_rho(crit_test, pred),
    }

    # Heuristics (if columns exist in original df)
    # Heuristic scores: larger => more critical
    heur = {}
    if HEUR_HPWL_COL in all_df.columns:
        hpwl = all_df.loc[test_order, HEUR_HPWL_COL].to_numpy()
        heur["Heur_HPWL_Recall@K"] = recall_at_k(slack_test, hpwl, K)
        heur["Heur_HPWL_Spearman"] = spearman_rho(crit_test, hpwl)
    if HEUR_FANOUT_COL in all_df.columns:
        fan = all_df.loc[test_order, HEUR_FANOUT_COL].to_numpy()
        heur["Heur_Fanout_Recall@K"] = recall_at_k(slack_test, fan, K)
        heur["Heur_Fanout_Spearman"] = spearman_rho(crit_test, fan)

    res.update(heur)

    return res


# ----------------------------
# 7) MAIN: LOBO + Ablations
# ----------------------------
all_df = load_blocks(BLOCKS, DATASET_ROOT)
all_df, X_all = build_feature_matrix(all_df)

print("Loaded blocks:", BLOCKS)
print("Total endpoints:", len(all_df))
print("Feature dim (after OHE):", X_all.shape[1])
print("Columns in df:", list(all_df.columns))


# ---- (A) Leave-One-Block-Out Table (for paper) ----
lobo_rows = []
for test_blk in BLOCKS:
    train_blks = [b for b in BLOCKS if b != test_blk]
    # Full features
    lobo_rows.append(run_experiment(all_df, X_all, train_blks, [test_blk], feature_mode="full"))
    # No spatial
    lobo_rows.append(run_experiment(all_df, X_all, train_blks, [test_blk], feature_mode="no_spatial"))
    # Baseline only
    lobo_rows.append(run_experiment(all_df, X_all, train_blks, [test_blk], feature_mode="only_baseline"))

lobo_df = pd.DataFrame(lobo_rows)

# Reorder columns for readability
front = ["test_blocks", "train_blocks", "feature_mode",
         "N_test_endpoints", "N_test_groups", "K_violating",
         "Recall@K", "NDCG@K", "Spearman"]
rest = [c for c in lobo_df.columns if c not in front]
lobo_df = lobo_df[front + rest].sort_values(["test_blocks", "feature_mode"])

print("\n=== LOBO results (table for paper) ===")
display(lobo_df)

if SAVE_RESULTS:
    lobo_df.to_csv(os.path.join(OUT_DIR, "lobo_table.csv"), index=False)


# ---- (B) One custom split (train/test lists you choose) ----
# Example:
TRAIN_BLOCKS = ["pcm_slv_top", "FIR_filter"]
TEST_BLOCKS  = ["spi_top"]

custom_rows = []
for mode in ["full", "no_spatial", "only_baseline"]:
    custom_rows.append(run_experiment(all_df, X_all, TRAIN_BLOCKS, TEST_BLOCKS, feature_mode=mode))
custom_df = pd.DataFrame(custom_rows)

print("\n=== Custom split results ===")
display(custom_df)

if SAVE_RESULTS:
    custom_df.to_csv(os.path.join(OUT_DIR, "custom_split_table.csv"), index=False)


# ----------------------------
# 8) Produce a Markdown table snippet (ready for paper)
# ----------------------------
def to_paper_markdown(df: pd.DataFrame,
                      cols=("test_blocks","feature_mode","Recall@K","NDCG@K","Spearman")) -> str:
    d = df[list(cols)].copy()
    # nice formatting
    for c in ["Recall@K","NDCG@K","Spearman"]:
        if c in d.columns:
            d[c] = d[c].map(lambda x: f"{x:.3f}" if pd.notna(x) else "NA")
    return d.to_markdown(index=False)

paper_md = to_paper_markdown(lobo_df)
print("\n=== Markdown table (paste into paper draft) ===\n")
print(paper_md)

if SAVE_RESULTS:
    with open(os.path.join(OUT_DIR, "lobo_table.md"), "w", encoding="utf-8") as f:
        f.write(paper_md)

# ----------------------------
# 9) Notes for the paper (Methodology snippets)
# ----------------------------
NOTES_FOR_PAPER = r"""
Evaluation protocol:
- We treat timing prediction as learning-to-rank over sequential endpoints.
- Ranking groups correspond to (block, clock domain), preventing mixing different timing contexts.
- We report Recall@K where K equals the number of violating endpoints (slack < 0) in the test set,
  as well as NDCG@K and Spearman correlation between predicted scores and post-route (-slack).

Ablation study:
- Full: all place-stage endpoint features.
- No-spatial: removes spatial neighborhood features (e.g., ff_neighbors_r20/r50/r100).
- Baseline-only: retains only simple wirelength/fanout proxies and basic net statistics.
"""

print("\n=== Paper notes ===\n", NOTES_FOR_PAPER)

if SAVE_RESULTS:
    with open(os.path.join(OUT_DIR, "paper_notes.txt"), "w", encoding="utf-8") as f:
        f.write(NOTES_FOR_PAPER)


Loaded blocks: ['pcm_slv_top', 'FIR_filter', 'spi_top', 'usb_phy']
Total endpoints: 809
Feature dim (after OHE): 50
Columns in df: ['endpoint', 'instance', 'worst_pin_regex', 'ff_area', 'ff_ref', 'ff_x', 'ff_y', 'ff_w', 'ff_h', 'ff_cx', 'ff_cy', 'ff_orient', 'ff_status', 'ff_in_pins', 'ff_out_pins', 'ff_clk_pins', 'ff_inout_pins', 'D_net', 'D_is_port', 'D_dx', 'D_dy', 'D_hpwl', 'D_bbox_area', 'D_termCount', 'D_iterms', 'D_bterms', 'D_fanout_proxy', 'D_net_center_dx', 'D_net_center_dy', 'in_nets_cnt', 'in_hpwl_sum', 'in_hpwl_mean', 'in_hpwl_max', 'in_fanout_mean', 'in_fanout_max', 'in_term_mean', 'in_term_max', 'in_ports_cnt', 'ff_neighbors_r20', 'ff_neighbors_r50', 'ff_neighbors_r100', 'worst_slack_ns', 'endpoint_clock', 'block_name', 'crit_cont']


/home/nzheludkov/phd/ml-critical-path-prediction/env/lib/python3.12/site-packages/pandas/core/nanops.py:1661: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/home/nzheludkov/phd/ml-critical-path-prediction/env/lib/python3.12/site-packages/pandas/core/nanops.py:1661: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/home/nzheludkov/phd/ml-critical-path-prediction/env/lib/python3.12/site-packages/pandas/core/nanops.py:1661: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]



=== LOBO results (table for paper) ===


,test_blocks,train_blocks,feature_mode,N_test_endpoints,N_test_groups,K_violating,Recall@K,NDCG@K,Spearman,Heur_HPWL_Recall@K,Heur_HPWL_Spearman,Heur_Fanout_Recall@K,Heur_Fanout_Spearman
3,FIR_filter,"pcm_slv_top,spi_top,usb_phy",full,407,1,12,0.083333,-1.507728,-0.028542,0.000000,-0.117483,0.000000,-0.266648
4,FIR_filter,"pcm_slv_top,spi_top,usb_phy",no_spatial,407,1,12,0.250000,-0.559560,0.062959,0.000000,-0.117483,0.000000,-0.266648
5,FIR_filter,"pcm_slv_top,spi_top,usb_phy",only_baseline,407,1,12,0.000000,-1.791274,0.091568,0.000000,-0.117483,0.000000,-0.266648
0,pcm_slv_top,"FIR_filter,spi_top,usb_phy",full,68,1,1,0.000000,-4.680271,0.159369,0.000000,-0.441308,0.000000,-0.540554
1,pcm_slv_top,"FIR_filter,spi_top,usb_phy",no_spatial,68,1,1,0.000000,-4.680271,0.138756,0.000000,-0.441308,0.000000,-0.540554
2,pcm_slv_top,"FIR_filter,spi_top,usb_phy",only_baseline,68,1,1,0.000000,-6.238857,-0.256902,0.000000,-0.441308,0.000000,-0.540554
6,spi_top,"pcm_slv_top,FIR_filter,usb_phy",full,229,1,74,0.229730,-3.983840,-0.067086,0.202703,-0.311953,0.554054,NaN
7,spi_top,"pcm_slv_top,FIR_filter,usb_phy",no_spatial,229,1,74,0.216216,-4.518595,-0.118402,0.202703,-0.311953,0.554054,NaN
8,spi_top,"pcm_slv_top,FIR_filter,usb_phy",only_baseline,229,1,74,0.364865,-2.466600,0.101982,0.202703,-0.311953,0.554054,NaN
9,usb_phy,"pcm_slv_top,FIR_filter,spi_top",full,105,1,7,0.285714,-2.068103,0.342194,0.000000,-0.263976,0.000000,-0.591726


/home/nzheludkov/phd/ml-critical-path-prediction/env/lib/python3.12/site-packages/pandas/core/nanops.py:1661: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


KeyboardInterrupt: 

In [11]:
# ============================================================
# Minimal XGBoost LTR baseline for endpoint timing criticality
# - Multi-block load (each block in its own folder)
# - Train/Test block selection
# - Ranking groups = (block, endpoint_clock)
# - Metrics: Recall@K, NDCG@K, Spearman
# ============================================================

import os
import numpy as np
import pandas as pd
from xgboost import XGBRanker

# ----------------------------
# CONFIG
# ----------------------------
DATASET_ROOT = "/home/nzheludkov/phd/ml-critical-path-prediction/dataset"

TRAIN_BLOCKS = ["pcm_slv_top", "FIR_filter", "usb_phy", "simple_spi_top", "sasc_top" , "i2c_master_top", "ac97_top", "mc_top", "sha256"]

TEST_BLOCKS  = ["spi_top"]

FEAT_REL = os.path.join("features", "endpoint_features_place.csv")
LAB_REL  = os.path.join("features", "endpoint_slacks_postroute.csv")

CAT_COLS = ["ff_ref", "ff_orient", "ff_status"]  # one-hot if present

DROP_ALWAYS = [
    "endpoint", "instance", "D_net", "worst_pin_regex",
    "worst_slack_ns", "crit_cont",
    "block_name", "endpoint_clock"
]

# XGBoost params (start point)
XGB_PARAMS = dict(
    objective="rank:pairwise",
    n_estimators=600,
    learning_rate=0.06,
    max_depth=5,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    tree_method="hist",
    eval_metric="ndcg",
    random_state=42,
    n_jobs=8,
)

# IMPORTANT FIX:
# XGBoost's NDCG with exponential gain requires labels <= 31
MAX_GRADE = 31


# ----------------------------
# LOADERS
# ----------------------------
def load_block(block_name: str, root: str) -> pd.DataFrame:
    feat_path = os.path.join(root, block_name, FEAT_REL)
    lab_path  = os.path.join(root, block_name, LAB_REL)

    feat = pd.read_csv(feat_path)
    lab  = pd.read_csv(lab_path)

    if "endpoint" not in lab.columns or "worst_slack_ns" not in lab.columns:
        raise ValueError(f"[{block_name}] labels csv must contain endpoint + worst_slack_ns")

    # if endpoint_clock missing, fallback to single domain
    if "endpoint_clock" not in lab.columns:
        lab = lab.copy()
        lab["endpoint_clock"] = "clk0"

    df = feat.merge(
        lab[["endpoint", "worst_slack_ns", "endpoint_clock"]],
        on="endpoint",
        how="inner"
    )
    df["block_name"] = block_name
    return df


def load_blocks(block_list, root) -> pd.DataFrame:
    dfs = [load_block(b, root) for b in block_list]
    return pd.concat(dfs, ignore_index=True)


# ----------------------------
# FEATURES (global)
# ----------------------------
def build_feature_matrix(all_df: pd.DataFrame):
    all_df = all_df.copy()
    all_df["crit_cont"] = -all_df["worst_slack_ns"]  # higher => more critical

    drop_cols = [c for c in DROP_ALWAYS if c in all_df.columns]
    base = all_df.drop(columns=drop_cols, errors="ignore")

    cat_cols = [c for c in CAT_COLS if c in base.columns]
    X_all = pd.get_dummies(base, columns=cat_cols, dummy_na=True)

    return all_df, X_all


# ----------------------------
# GROUPS for ranking
# ----------------------------
def build_groups_and_order(df_sub: pd.DataFrame):
    groups = []
    order = []
    for (_, _), g in df_sub.groupby(["block_name", "endpoint_clock"], sort=False):
        groups.append(len(g))
        order.extend(g.index.tolist())
    return groups, order


# ----------------------------
# Convert continuous -> integer grades (0..MAX_GRADE) per group
# (required for XGB >= 3.x + NDCG exp gain)
# ----------------------------
def to_grades_per_group(crit_vec: np.ndarray, groups: list[int], max_grade: int = 31) -> np.ndarray:
    grades = np.empty_like(crit_vec, dtype=np.int32)
    start = 0
    for g in groups:
        end = start + g
        c = crit_vec[start:end]

        # rank positions: 0 least critical ... g-1 most critical
        order = np.argsort(c)
        r = np.empty(g, dtype=np.int32)
        r[order] = np.arange(g, dtype=np.int32)

        if g == 1:
            grades[start:end] = 0
        else:
            grades[start:end] = np.floor(r * (max_grade / (g - 1))).astype(np.int32)

        start = end
    return grades


# ----------------------------
# METRICS
# ----------------------------
def recall_at_k(slack_true: np.ndarray, score: np.ndarray, k: int) -> float:
    if k <= 0:
        return float("nan")
    violating = (slack_true < 0).astype(int)
    denom = int(violating.sum())
    if denom == 0:
        return float("nan")
    order = np.argsort(-score)[:k]
    return float(violating[order].sum() / denom)


def spearman_rho(a: np.ndarray, b: np.ndarray) -> float:
    return float(pd.Series(a).corr(pd.Series(b), method="spearman"))


def ndcg_at_k(rel_true: np.ndarray, score: np.ndarray, k: int) -> float:
    # rel_true: higher => more relevant/critical; here we use continuous crit_cont for evaluation
    if k <= 0:
        return float("nan")
    k = min(k, len(score))
    order = np.argsort(-score)[:k]
    rel = rel_true[order].astype(float)

    gains = (2.0 ** rel - 1.0)
    discounts = 1.0 / np.log2(np.arange(2, 2 + k))
    dcg = float(np.sum(gains * discounts))

    ideal = np.argsort(-rel_true)[:k]
    rel_i = rel_true[ideal].astype(float)
    gains_i = (2.0 ** rel_i - 1.0)
    idcg = float(np.sum(gains_i * discounts))

    return float(dcg / idcg) if idcg > 0 else float("nan")


# ============================================================
# RUN
# ============================================================
train_df = load_blocks(TRAIN_BLOCKS, DATASET_ROOT)
test_df  = load_blocks(TEST_BLOCKS,  DATASET_ROOT)

all_df = pd.concat([train_df, test_df], ignore_index=True)
all_df, X_all = build_feature_matrix(all_df)

# Split indices back (by row counts)
train_idx = np.arange(len(train_df))
test_idx  = np.arange(len(train_df), len(train_df) + len(test_df))

train_df2 = all_df.iloc[train_idx].copy()
test_df2  = all_df.iloc[test_idx].copy()

# Build group ordering (concatenate groups)
train_groups, train_order = build_groups_and_order(train_df2)
test_groups,  test_order  = build_groups_and_order(test_df2)

# Align features/targets
X_train = X_all.loc[train_order].to_numpy()
X_test  = X_all.loc[test_order].to_numpy()

crit_train = all_df.loc[train_order, "crit_cont"].to_numpy()
crit_test  = all_df.loc[test_order,  "crit_cont"].to_numpy()
slack_test = all_df.loc[test_order,  "worst_slack_ns"].to_numpy()

y_train = to_grades_per_group(crit_train, train_groups, MAX_GRADE)
y_test  = to_grades_per_group(crit_test,  test_groups,  MAX_GRADE)

# Train
ranker = XGBRanker(**XGB_PARAMS)
ranker.fit(
    X_train, y_train,
    group=train_groups,
    eval_set=[(X_test, y_test)],
    eval_group=[test_groups],
    verbose=False
)

# Predict + evaluate
pred = ranker.predict(X_test)
K = int((slack_test < 0).sum())

print("===================================")
print("XGBoost LTR baseline (endpoints)")
print("Train blocks:", TRAIN_BLOCKS)
print("Test blocks :", TEST_BLOCKS)
print("-----------------------------------")
print("Train endpoints:", len(train_df))
print("Test endpoints :", len(test_df))
print("Test groups    :", len(test_groups), "(block,clock)")
print("Violations in test (slack<0): K =", K)
print("-----------------------------------")
print("Recall@K :", recall_at_k(slack_test, pred, K))
print("NDCG@K   :", ndcg_at_k(crit_test, pred, max(K, 1)))
print("Spearman :", spearman_rho(crit_test, pred))
print("===================================")


XGBoost LTR baseline (endpoints)
Train blocks: ['pcm_slv_top', 'FIR_filter', 'usb_phy', 'simple_spi_top', 'sasc_top', 'i2c_master_top', 'ac97_top', 'mc_top', 'sha256']
Test blocks : ['spi_top']
-----------------------------------
Train endpoints: 4369
Test endpoints : 229
Test groups    : 1 (block,clock)
Violations in test (slack<0): K = 74
-----------------------------------
Recall@K : 0.25675675675675674
NDCG@K   : -4.22313967988348
Spearman : -0.21469427318057824


In [13]:
# ============================================================
# OPTIONAL: Leave-One-Block-Out (LOBO) loop
# ============================================================

DO_LOBO = True   # <-- поставь False, если хочешь только один кастомный split

ALL_BLOCKS = ["pcm_slv_top", "FIR_filter", "usb_phy", "simple_spi_top", "sasc_top" , "i2c_master_top", "ac97_top", "mc_top", "sha256"]  # список всех доступных блоков

def run_train_test(train_blocks, test_blocks):
    train_df = load_blocks(train_blocks, DATASET_ROOT)
    test_df  = load_blocks(test_blocks,  DATASET_ROOT)

    all_df = pd.concat([train_df, test_df], ignore_index=True)
    all_df, X_all = build_feature_matrix(all_df)

    train_idx = np.arange(len(train_df))
    test_idx  = np.arange(len(train_df), len(train_df) + len(test_df))

    train_df2 = all_df.iloc[train_idx].copy()
    test_df2  = all_df.iloc[test_idx].copy()

    train_groups, train_order = build_groups_and_order(train_df2)
    test_groups,  test_order  = build_groups_and_order(test_df2)

    X_train = X_all.loc[train_order].to_numpy()
    X_test  = X_all.loc[test_order].to_numpy()

    crit_train = all_df.loc[train_order, "crit_cont"].to_numpy()
    crit_test  = all_df.loc[test_order,  "crit_cont"].to_numpy()
    slack_test = all_df.loc[test_order,  "worst_slack_ns"].to_numpy()

    y_train = to_grades_per_group(crit_train, train_groups, MAX_GRADE)
    y_test  = to_grades_per_group(crit_test,  test_groups,  MAX_GRADE)

    ranker = XGBRanker(**XGB_PARAMS)
    ranker.fit(
        X_train, y_train,
        group=train_groups,
        eval_set=[(X_test, y_test)],
        eval_group=[test_groups],
        verbose=False
    )

    pred = ranker.predict(X_test)
    K = int((slack_test < 0).sum())

    return {
        "train_blocks": ",".join(train_blocks),
        "test_blocks": ",".join(test_blocks),
        "test_endpoints": int(len(test_df)),
        "test_groups": int(len(test_groups)),
        "K_viol": int(K),
        "Recall@K": recall_at_k(slack_test, pred, K),
        "NDCG@K": ndcg_at_k(crit_test, pred, max(K, 1)),
        "Spearman": spearman_rho(crit_test, pred),
    }


if DO_LOBO:
    print("\n==================== LOBO ====================")
    rows = []
    for test_blk in ALL_BLOCKS:
        train_blks = [b for b in ALL_BLOCKS if b != test_blk]
        res = run_train_test(train_blks, [test_blk])
        rows.append(res)

        print(f"\nTest block: {test_blk}")
        print("  Train:", train_blks)
        print("  Test endpoints:", res["test_endpoints"], "groups:", res["test_groups"], "K:", res["K_viol"])
        print("  Recall@K :", res["Recall@K"])
        print("  NDCG@K   :", res["NDCG@K"])
        print("  Spearman :", res["Spearman"])

    lobo_df = pd.DataFrame(rows)
    print("\n================ LOBO SUMMARY ================")
    display(lobo_df)



==================== LOBO ====================

Test block: pcm_slv_top
  Train: ['FIR_filter', 'usb_phy', 'simple_spi_top', 'sasc_top', 'i2c_master_top', 'ac97_top', 'mc_top', 'sha256']
  Test endpoints: 68 groups: 1 K: 1
  Recall@K : 0.0
  NDCG@K   : -4.680270985041086
  Spearman : 0.4608924686032751

Test block: FIR_filter
  Train: ['pcm_slv_top', 'usb_phy', 'simple_spi_top', 'sasc_top', 'i2c_master_top', 'ac97_top', 'mc_top', 'sha256']
  Test endpoints: 407 groups: 1 K: 12
  Recall@K : 0.5
  NDCG@K   : 0.278220123183033
  Spearman : 0.2259119779403755

Test block: usb_phy
  Train: ['pcm_slv_top', 'FIR_filter', 'simple_spi_top', 'sasc_top', 'i2c_master_top', 'ac97_top', 'mc_top', 'sha256']
  Test endpoints: 105 groups: 1 K: 7
  Recall@K : 0.14285714285714285
  NDCG@K   : -2.659070980740726
  Spearman : 0.4581691893012647

Test block: simple_spi_top
  Train: ['pcm_slv_top', 'FIR_filter', 'usb_phy', 'sasc_top', 'i2c_master_top', 'ac97_top', 'mc_top', 'sha256']
  Test endpoints: 99 gr

,train_blocks,test_blocks,test_endpoints,test_groups,K_viol,Recall@K,NDCG@K,Spearman
0,"FIR_filter,usb_phy,simple_spi_top,sasc_top,i2c...",pcm_slv_top,68,1,1,0.000000,-4.680271,0.460892
1,"pcm_slv_top,usb_phy,simple_spi_top,sasc_top,i2...",FIR_filter,407,1,12,0.500000,0.278220,0.225912
2,"pcm_slv_top,FIR_filter,simple_spi_top,sasc_top...",usb_phy,105,1,7,0.142857,-2.659071,0.458169
3,"pcm_slv_top,FIR_filter,usb_phy,sasc_top,i2c_ma...",simple_spi_top,99,1,0,NaN,NaN,0.533790
4,"pcm_slv_top,FIR_filter,usb_phy,simple_spi_top,...",sasc_top,85,1,0,NaN,NaN,0.556149
5,"pcm_slv_top,FIR_filter,usb_phy,simple_spi_top,...",i2c_master_top,127,1,0,NaN,NaN,-0.246631
6,"pcm_slv_top,FIR_filter,usb_phy,simple_spi_top,...",ac97_top,1803,1,0,NaN,NaN,-0.063719
7,"pcm_slv_top,FIR_filter,usb_phy,simple_spi_top,...",mc_top,601,1,181,0.436464,0.207553,0.437997
8,"pcm_slv_top,FIR_filter,usb_phy,simple_spi_top,...",sha256,1074,1,22,0.045455,-0.204309,0.592451


In [32]:
# ============================================================
# Minimal XGBoost LTR baseline (endpoints) with slack normalization
# - Per-block loading from folders
# - Uses CLK_PERIOD from metrics.csv to compute slack_norm
# - Train/Test block selection
# - Ranking groups = (block_name, endpoint_clock)
# - Metrics: Recall@top-p%, NDCG@top-p%, Spearman on crit_norm
# ============================================================

import os
import numpy as np
import pandas as pd
from xgboost import XGBRanker

# ----------------------------
# CONFIG
# ----------------------------
DATASET_ROOT = "/home/nzheludkov/phd/ml-critical-path-prediction/dataset"

TRAIN_BLOCKS = ["pcm_slv_top", "FIR_filter", "mc_top", "simple_spi_top", "i2c_master_top" , "usb_phy", "sha256", "ac97_top", "spi_top"]
TEST_BLOCKS  = ["sasc_top"]

FEAT_REL = os.path.join("features", "endpoint_features_place.csv")
LAB_REL  = os.path.join("features", "endpoint_slacks_postroute.csv")
MET_REL  = "metrics.csv"   # <-- per-block metrics.csv containing CLK_PERIOD

CAT_COLS = ["ff_ref", "ff_orient", "ff_status"]  # one-hot if present

DROP_ALWAYS = [
    "endpoint", "instance", "D_net", "worst_pin_regex",
    "worst_slack_ns", "CLK_PERIOD", "slack_norm", "crit_norm",
    "block_name", "endpoint_clock"
]

# Evaluate top-p% instead of K=#violating
TOP_P = 0.10  # 10%

# XGBoost params (start point)
XGB_PARAMS = dict(
    objective="rank:ndcg",
    n_estimators=600,
    learning_rate=0.06,
    max_depth=5,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    tree_method="hist",
    eval_metric="ndcg",
    random_state=42,
    n_jobs=8,
)

# IMPORTANT FIX:
# XGBoost's NDCG with exponential gain requires labels <= 31
MAX_GRADE = 31


# ----------------------------
# LOADERS
# ----------------------------
def load_clk_period(block_name: str, root: str) -> float:
    met_path = os.path.join(root, block_name, MET_REL)
    m = pd.read_csv(met_path, sep=";")
    if "CLK_PERIOD" not in m.columns:
        raise ValueError(f"[{block_name}] metrics.csv missing CLK_PERIOD column")
    # assume single value for block (first row)
    return float(m["CLK_PERIOD"].iloc[0])


def load_block(block_name: str, root: str) -> pd.DataFrame:
    feat_path = os.path.join(root, block_name, FEAT_REL)
    lab_path  = os.path.join(root, block_name, LAB_REL)

    feat = pd.read_csv(feat_path)
    lab  = pd.read_csv(lab_path)

    if "endpoint" not in lab.columns or "worst_slack_ns" not in lab.columns:
        raise ValueError(f"[{block_name}] labels csv must contain endpoint + worst_slack_ns")

    # if endpoint_clock missing, fallback to single domain
    if "endpoint_clock" not in lab.columns:
        lab = lab.copy()
        lab["endpoint_clock"] = "clk0"

    df = feat.merge(
        lab[["endpoint", "worst_slack_ns", "endpoint_clock"]],
        on="endpoint",
        how="inner"
    )
    df["block_name"] = block_name

    # add CLK_PERIOD from metrics.csv
    clk_period = load_clk_period(block_name, root)
    df["CLK_PERIOD"] = clk_period

    # normalized target
    df["slack_norm"] = df["worst_slack_ns"] / df["CLK_PERIOD"]
    df["crit_norm"]  = -df["slack_norm"]  # higher => more critical

    return df


def load_blocks(block_list, root) -> pd.DataFrame:
    dfs = [load_block(b, root) for b in block_list]
    return pd.concat(dfs, ignore_index=True)

def add_block_normalized_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add per-block normalized features to reduce cross-block scale shift.
    Assumes df contains block_name and placement bbox information.
    """

    df = df.copy()

    # --- block-level stats ---
    # block bbox from FF placement (robust and always available)
    block_stats = []

    for blk, g in df.groupby("block_name"):
        xs = g["ff_cx"].to_numpy() if "ff_cx" in g.columns else g["ff_x"].to_numpy()
        ys = g["ff_cy"].to_numpy() if "ff_cy" in g.columns else g["ff_y"].to_numpy()

        xmin, xmax = xs.min(), xs.max()
        ymin, ymax = ys.min(), ys.max()

        w = xmax - xmin
        h = ymax - ymin
        area = max(w * h, 1e-6)

        block_stats.append({
            "block_name": blk,
            "block_bbox_w": w,
            "block_bbox_h": h,
            "block_bbox_area": area,
            "num_ffs": len(g),
        })

    bs = pd.DataFrame(block_stats)
    df = df.merge(bs, on="block_name", how="left")

    # --- normalize geometry-like features ---
    geo_like = [
        c for c in df.columns
        if any(k in c.lower() for k in ["hpwl", "bbox", "dx", "dy", "dist", "offset"])
    ]

    for c in geo_like:
        df[c + "_norm"] = df[c] / np.sqrt(df["block_bbox_area"])

    # --- normalize count-like features ---
    count_like = [
        c for c in df.columns
        if any(k in c.lower() for k in ["fanout", "fanin", "neighbors", "term", "ports"])
    ]

    for c in count_like:
        df[c + "_norm"] = df[c] / np.maximum(df["num_ffs"], 1)

    return df


# ----------------------------
# FEATURES (global)
# ----------------------------
def build_feature_matrix(all_df: pd.DataFrame):
    all_df = all_df.copy()

    drop_cols = [c for c in DROP_ALWAYS if c in all_df.columns]
    base = all_df.drop(columns=drop_cols, errors="ignore")

    cat_cols = [c for c in CAT_COLS if c in base.columns]
    X_all = pd.get_dummies(base, columns=cat_cols, dummy_na=True)

    return all_df, X_all


# ----------------------------
# GROUPS for ranking
# ----------------------------
def build_groups_and_order(df_sub: pd.DataFrame):
    groups = []
    order = []
    for (_, _), g in df_sub.groupby(["block_name", "endpoint_clock"], sort=False):
        groups.append(len(g))
        order.extend(g.index.tolist())
    return groups, order


# ----------------------------
# Convert continuous crit_norm -> integer grades (0..MAX_GRADE) per group
# (required for XGB >= 3.x + NDCG exp gain)
# ----------------------------
def to_grades_per_group(crit_vec: np.ndarray, groups: list[int], max_grade: int = 31) -> np.ndarray:
    grades = np.empty_like(crit_vec, dtype=np.int32)
    start = 0
    for g in groups:
        end = start + g
        c = crit_vec[start:end]

        # rank positions: 0 least critical ... g-1 most critical
        order = np.argsort(c)
        r = np.empty(g, dtype=np.int32)
        r[order] = np.arange(g, dtype=np.int32)

        if g == 1:
            grades[start:end] = 0
        else:
            grades[start:end] = np.floor(r * (max_grade / (g - 1))).astype(np.int32)

        start = end
    return grades


# ----------------------------
# METRICS (top-p%)
# ----------------------------
def recall_at_top_p(crit_true: np.ndarray, score: np.ndarray, p: float) -> float:
    n = len(score)
    k = max(1, int(np.ceil(p * n)))
    gt = np.argsort(-crit_true)[:k]
    pr = np.argsort(-score)[:k]
    return float(len(set(gt) & set(pr)) / k)


def spearman_rho(a: np.ndarray, b: np.ndarray) -> float:
    return float(pd.Series(a).corr(pd.Series(b), method="spearman"))


def ndcg_at_top_p(crit_true: np.ndarray, score: np.ndarray, p: float) -> float:
    """
    NDCG@top-p% using crit_true as relevance (higher is more relevant/critical).
    Uses exponential gain (2^rel - 1) for consistency with common NDCG definition.
    """
    n = len(score)
    k = max(1, int(np.ceil(p * n)))
    k = min(k, n)

    order = np.argsort(-score)[:k]
    rel = crit_true[order].astype(float)

    gains = (2.0 ** rel - 1.0)
    discounts = 1.0 / np.log2(np.arange(2, 2 + k))
    dcg = float(np.sum(gains * discounts))

    ideal = np.argsort(-crit_true)[:k]
    rel_i = crit_true[ideal].astype(float)
    gains_i = (2.0 ** rel_i - 1.0)
    idcg = float(np.sum(gains_i * discounts))

    return float(dcg / idcg) if idcg > 0 else float("nan")


# ============================================================
# RUN
# ============================================================
train_df = load_blocks(TRAIN_BLOCKS, DATASET_ROOT)
test_df  = load_blocks(TEST_BLOCKS,  DATASET_ROOT)

all_df = pd.concat([train_df, test_df], ignore_index=True)

# ADD THIS LINE
all_df = add_block_normalized_features(all_df)

all_df, X_all = build_feature_matrix(all_df)


# Split indices back (by row counts)
train_idx = np.arange(len(train_df))
test_idx  = np.arange(len(train_df), len(train_df) + len(test_df))

train_df2 = all_df.iloc[train_idx].copy()
test_df2  = all_df.iloc[test_idx].copy()

# Build group ordering (concatenate groups)
train_groups, train_order = build_groups_and_order(train_df2)
test_groups,  test_order  = build_groups_and_order(test_df2)

# Align features/targets
X_train = X_all.loc[train_order].to_numpy()
X_test  = X_all.loc[test_order].to_numpy()

crit_train = all_df.loc[train_order, "crit_norm"].to_numpy()
crit_test  = all_df.loc[test_order,  "crit_norm"].to_numpy()

y_train = to_grades_per_group(crit_train, train_groups, MAX_GRADE)
y_test  = to_grades_per_group(crit_test,  test_groups,  MAX_GRADE)

# Train
ranker = XGBRanker(**XGB_PARAMS)
ranker.fit(
    X_train, y_train,
    group=train_groups,
    eval_set=[(X_test, y_test)],
    eval_group=[test_groups],
    verbose=False
)

# Predict + evaluate
pred = ranker.predict(X_test)

print("===================================")
print("XGBoost LTR baseline (endpoints, normalized slack)")
print("Train blocks:", TRAIN_BLOCKS)
print("Test blocks :", TEST_BLOCKS)
print("-----------------------------------")
print("Train endpoints:", len(train_df))
print("Test endpoints :", len(test_df))
print("Test groups    :", len(test_groups), "(block,clock)")
print("TOP_P =", TOP_P, "=> K =", max(1, int(np.ceil(TOP_P * len(test_df))),))
print("-----------------------------------")
print("Recall@top-p% :", recall_at_top_p(crit_test, pred, TOP_P))
print("NDCG@top-p%   :", ndcg_at_top_p(crit_test, pred, TOP_P))
print("Spearman      :", spearman_rho(crit_test, pred))
print("===================================")


XGBoost LTR baseline (endpoints, normalized slack)
Train blocks: ['pcm_slv_top', 'FIR_filter', 'mc_top', 'simple_spi_top', 'i2c_master_top', 'usb_phy', 'sha256', 'ac97_top', 'spi_top']
Test blocks : ['sasc_top']
-----------------------------------
Train endpoints: 4513
Test endpoints : 85
Test groups    : 1 (block,clock)
TOP_P = 0.1 => K = 9
-----------------------------------
Recall@top-p% : 0.1111111111111111
NDCG@top-p%   : nan
Spearman      : 0.5075386704970735


In [26]:
def run_intra_block_upper_bound(
    block_name: str,
    train_frac: float = 0.8,
    random_state: int = 42
):
    """
    Train/test split within ONE block to estimate upper bound.
    Ranking groups = (block_name, endpoint_clock).
    Evaluation = Recall@top-p%, NDCG@top-p%, Spearman on crit_norm.
    """

    # Load single block
    df = load_blocks([block_name], DATASET_ROOT)
    all_df, X_all = build_feature_matrix(df)

    # Random split by endpoint index
    rng = np.random.default_rng(random_state)
    idx = np.arange(len(all_df))
    rng.shuffle(idx)

    n_train = int(train_frac * len(idx))
    train_idx = idx[:n_train]
    test_idx  = idx[n_train:]

    train_df = all_df.iloc[train_idx].copy()
    test_df  = all_df.iloc[test_idx].copy()

    # Build ranking groups independently for train/test
    train_groups, train_order = build_groups_and_order(train_df)
    test_groups,  test_order  = build_groups_and_order(test_df)

    # Align feature matrices
    X_train = X_all.loc[train_order].to_numpy()
    X_test  = X_all.loc[test_order].to_numpy()

    crit_train = all_df.loc[train_order, "crit_norm"].to_numpy()
    crit_test  = all_df.loc[test_order,  "crit_norm"].to_numpy()

    # Convert to integer grades for XGB
    y_train = to_grades_per_group(crit_train, train_groups, MAX_GRADE)
    y_test  = to_grades_per_group(crit_test,  test_groups,  MAX_GRADE)

    # Train ranker
    ranker = XGBRanker(**XGB_PARAMS)
    ranker.fit(
        X_train, y_train,
        group=train_groups,
        eval_set=[(X_test, y_test)],
        eval_group=[test_groups],
        verbose=False
    )

    # Predict + metrics
    pred = ranker.predict(X_test)

    recall = recall_at_top_p(crit_test, pred, TOP_P)
    ndcg   = ndcg_at_top_p(crit_test, pred, TOP_P)
    rho    = spearman_rho(crit_test, pred)

    print("===================================")
    print("INTRA-BLOCK UPPER BOUND")
    print("Block:", block_name)
    print("Endpoints:", len(all_df))
    print("Train/Test:", len(train_df), "/", len(test_df))
    print("Groups (train/test):", len(train_groups), "/", len(test_groups))
    print("-----------------------------------")
    print(f"Recall@top-{int(TOP_P*100)}% :", recall)
    print(f"NDCG@top-{int(TOP_P*100)}%   :", ndcg)
    print("Spearman              :", rho)
    print("===================================")

    return {
        "block": block_name,
        "Recall@top-p%": recall,
        "NDCG@top-p%": ndcg,
        "Spearman": rho,
    }


In [31]:
for blk in ["pcm_slv_top", "FIR_filter", "mc_top", "simple_spi_top", "sasc_top" , "i2c_master_top", "sha256", "ac97_top", "spi_top"]:
    run_intra_block_upper_bound(blk)

INTRA-BLOCK UPPER BOUND
Block: pcm_slv_top
Endpoints: 68
Train/Test: 54 / 14
Groups (train/test): 1 / 1
-----------------------------------
Recall@top-10% : 0.0
NDCG@top-10%   : nan
Spearman              : 0.2527472527472528
INTRA-BLOCK UPPER BOUND
Block: FIR_filter
Endpoints: 407
Train/Test: 325 / 82
Groups (train/test): 1 / 1
-----------------------------------
Recall@top-10% : 0.5555555555555556
NDCG@top-10%   : nan
Spearman              : 0.24231671540162028
INTRA-BLOCK UPPER BOUND
Block: mc_top
Endpoints: 601
Train/Test: 480 / 121
Groups (train/test): 1 / 1
-----------------------------------
Recall@top-10% : 0.5384615384615384
NDCG@top-10%   : 0.6454776959435385
Spearman              : 0.39703003237518236
INTRA-BLOCK UPPER BOUND
Block: simple_spi_top
Endpoints: 99
Train/Test: 79 / 20
Groups (train/test): 1 / 1
-----------------------------------
Recall@top-10% : 0.5
NDCG@top-10%   : nan
Spearman              : 0.613533834586466
INTRA-BLOCK UPPER BOUND
Block: sasc_top
Endpoints: 8